# بررسی تصویری چرخش GT — بدون تغییر دیتاست

این نوتبوک چهار فرضیه ثابت را با چرخش تخمین‌زده‌شده از تصاویر مقایسه می‌کند. نمونه‌های اولیه با شناسهٔ ۰ تا ۵ از ارزیابی حذف می‌شوند. از ابتدا، وسط و انتهای مسیر گروه‌هایی با مرکز ثبت‌شده یکسان انتخاب می‌کنیم.

**این آزمایش فقط چرخش نسبی را بررسی می‌کند؛ صحت انتقال، زمان ثبت یا کالیبراسیون مطلق را تضمین نمی‌کند.** کیفیت نامناسب تطبیق تصویر به معنی نتیجهٔ نامشخص است، نه خراب بودن GT.

پیش از اجرا، در محیط پایتون همین نوتبوک نصب کنید: `python -m pip install -r requirements-diagnostics.txt`

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
SEQUENCE = ROOT / 'standard_GT' / 'city01_route01'
# Original, headerless capture CSV, NOT the exported poses.csv:
SOURCE_CSV = ROOT / 'dataset' / 'positions_angles_camera1.csv'
GROUPS_PER_REGION = 2
SEED = 17
assert SEQUENCE.is_dir(), f'Edit SEQUENCE: {SEQUENCE}'
assert SOURCE_CSV.is_file(), f'Edit SOURCE_CSV: {SOURCE_CSV}'


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from diagnostics.rotation_check import load_inputs, choose_groups, run_diagnostic, save_report, HYPOTHESES
ds, source, rows = load_inputs(SEQUENCE, SOURCE_CSV)
groups = choose_groups(ds, source, rows, GROUPS_PER_REGION)
selection = [{'region': g['region'], 'exported_frames': g['frames'],
              'source_ids': [int(ds.rows[k]['source_frame_id']) for k in g['frames']],
              'csv_rows_zero_based': rows[g['frames']].tolist()} for g in groups]
display(pd.DataFrame(selection))
missing = {'beginning', 'middle', 'end'} - {g['region'] for g in groups}
print('Missing regions:', sorted(missing))
assert groups, 'No eligible groups. Do not replace these with moving pairs without checking parallax.'


### اجرای بررسی

برای هر جفت، تطبیق ویژگی‌های SIFT و برازش مقاوم چرخش انجام می‌شود. ۳۰٪ تطابق‌ها در برازش استفاده نمی‌شوند و برای بررسی مستقل کیفیت باقی می‌مانند. در صورت امکان، سازگاری دو نیمهٔ پانوراما نیز بررسی می‌شود.

`current_gt`: GT فعلی؛ `offset_only`: سطر قبلی CSV؛ `axes_only`: قرارداد محور پیشنهادی؛ `offset_and_axes`: ترکیب هر دو. جابه‌جایی سطر روی CSV اصلی است، نه فهرست عکس‌های باقی‌مانده.

In [ ]:
report = run_diagnostic(SEQUENCE, SOURCE_CSV, GROUPS_PER_REGION, SEED)
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT = ROOT / 'outputs' / 'gt_rotation_diagnostic' / SEQUENCE.name / stamp
save_report(report, OUT)
print('Report saved:', OUT)


In [ ]:
df = pd.DataFrame(report['results'])
error_columns = [h + '_error_deg' for h in HYPOTHESES]
for col in error_columns:
    if col not in df: df[col] = float('nan')
view = ['region', 'group', 'source_id_i', 'source_id_j', 'status',
        'heldout_inlier_fraction', 'hemisphere_disagreement_deg'] + error_columns
display(df.reindex(columns=view))
usable = df.loc[df['status'].eq('usable')].copy()
print(f'Usable pairs: {len(usable)}/{len(df)}; independent sampled groups: {usable.group.nunique()}')
print('Regions without selected groups:', report['regions_missing'])
if len(usable):
    summary = usable.groupby('region')[error_columns].agg(['count', 'median', 'max'])
    display(summary)
    summary.to_csv(OUT / 'summary.csv')
else:
    print('INCONCLUSIVE: no pair passed image consistency checks.')


In [ ]:
if len(usable):
    fig, ax = plt.subplots(figsize=(12, 5))
    labels = [f"{r.region}: {r.source_id_i}→{r.source_id_j}" for r in usable.itertuples()]
    for h in HYPOTHESES:
        ax.plot(range(len(usable)), usable[h + '_error_deg'], 'o-', label=h)
    ax.set_xticks(range(len(usable)), labels, rotation=65, ha='right')
    ax.set_ylabel('Relative rotation geodesic error (degrees)')
    ax.set_title('Only pairs passing image consistency checks')
    ax.legend(); ax.grid(alpha=.25); fig.tight_layout()
    fig.savefig(OUT / 'rotation_errors.png', dpi=160)
    plt.show()
import shutil
archive = shutil.make_archive(str(OUT.parent / (OUT.name + '_report')), 'zip', OUT)
print('Send this ZIP for interpretation:', archive)


### تفسیر نتیجه

- خطای کمتر فرضیهٔ ترکیبی در چند گروه مستقل، شواهدی به نفع ناسازگاری قرارداد محور/برچسب چرخش است؛ اثبات صحت کل دیتاست نیست.
- اختلاف دو نیمهٔ بیش از ۱ درجه نتیجه را نامشخص می‌کند. خالی بودن این مقدار یعنی یکی از نیمه‌ها تطابق کافی نداشته است؛ تأیید سازگاری دو نیمه محسوب نمی‌شود.
- سه جفت یک گروه مستقل از هم نیستند؛ تعداد گروه‌ها و پوشش مسیر را هم در نظر بگیرید.
- ثابت بودن مرکز از روی متادیتا بررسی شده؛ مرکز واقعی ثبت تصویر هنوز مستقلاً تأیید نشده است.
- خروجی این نوتبوک هیچ اصلاحی روی GT اعمال نمی‌کند. فایل ZIP شامل گزارش و نمودار است؛ تصاویر اصلی در آن نیستند.
